# **Split-origin sensitivity analysis**

In [1]:
"""
Split-origin sensitivity analysis  —  Reviewer #1, Concern #1.

Tests whether evaluation instances drawn from the ArabicaQA *train* split behave
differently from those drawn from the *test* split. If merging the partitions had
inflated results, train-origin instances would outperform test-origin ones.

The normalisation, EM, EM25, and abstention functions below are transcribed from
notebook 03_EM_EM25_ans-abstention-rate.ipynb and reproduce the published LLaMA
values exactly: EM 3.20, EM25 5.00, token-F1 0.1757, strict abstention 7.60,
flexible abstention 12.80, flexible false abstention 19.60. The VALIDATION block
re-checks this for every model before any breakdown is reported.

POWER WARNING
-------------
The test-origin answerable cell holds ~78 instances. At an EM rate near 3% the
permutation null is extremely discrete: for LLaMA it takes only 10 distinct
values, and the smallest non-zero difference on that grid (0.00601) equals the
observed difference, which forces p = 1.000. Report EM as *not testable at this
sample size*, never as evidence of no difference. token-F1, being continuous, is
the metric that actually carries information here.
"""

import ast
import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd

PRED_DIR = Path("arabicaqa_rag_results/predictions")
OUT_DIR = Path("arabicaqa_rag_results/split_sensitivity")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {"llama": "LLaMA 3 8B Instruct",
          "command": "Command-R7B-12-2024",
          "mistral": "Mistral-7B-Instruct-v0.1"}

# Published values: (ans EM %, ans EM25 %, token F1, strict abst %, flex abst %)
PUBLISHED = {
    "llama":   (3.20, 5.00, 0.1757, 7.60, 12.80),
    "command": (1.20, 2.20, 0.1625, 53.80, 54.20),
    "mistral": (0.60, 0.80, 0.0863, 7.60, 56.60),
}

NO_ANSWER = "غير موجود في السياق"
B = 10_000
SEED = 42


# ---------------------------------------------------------------- normalisation
def normalize_arabic_text(s):
    if s is None:
        return ""
    if isinstance(s, float) and math.isnan(s):
        return ""
    s = str(s).strip()
    s = re.sub(r"[\u064B-\u065F\u0670]", "", s)          # diacritics
    s = re.sub(r"[إأآا]", "ا", s)
    s = re.sub(r"ى", "ي", s)
    s = re.sub(r"ؤ", "و", s)
    s = re.sub(r"ئ", "ي", s)
    s = re.sub(r"ة", "ه", s)
    s = re.sub(r"ـ", "", s)                              # tatweel
    s = re.sub(r"[^\w\s\u0600-\u06FF]", " ", s)          # punctuation/symbols
    return re.sub(r"\s+", " ", s).strip()


NO_ANSWER_PATTERNS = [
    r"غير\s+موجود", r"غير\s+مذكور", r"غير\s+متوفر", r"ليس\s+مذكور(?:ا)?",
    r"لا\s+يوجد", r"لا\s+توجد", r"لا\s+تتوفر",
    r"لا\s+يحتوي\s+السياق", r"لم\s+يكن\s+(?:موجود(?:ا)?\s+)?في\s+السياق",
    r"لا\s+توجد\s+معلومات", r"المعلومات\s+غير\s+متوفر(?:ه|ة)",
    r"لا\s+يوجد\s+جواب", r"لا\s+يوجد\s+اجابه",
    r"لا\s+يمكن(?:ني)?\s+ال?اجابه", r"لا\s+يمكن(?:ني)?\s+تحديد",
    r"لا\s+يمكن(?:ني)?\s+العثور",
    r"لا\s+اعلم", r"لا\s+اعرف",
    r"not\s+found", r"not\s+mentioned", r"not\s+available",
    r"not\s+present\s+in\s+the\s+context", r"cannot\s+answer", r"i\s+do\s+not\s+know",
]


def is_flexible_no_answer(text):
    t = normalize_arabic_text(text)
    return bool(t) and any(re.search(p, t) for p in NO_ANSWER_PATTERNS)


def parse_answers(x):
    if isinstance(x, str):
        s = x.strip()
        if s.startswith("["):
            for loader in (json.loads, ast.literal_eval):
                try:
                    v = loader(s)
                    if isinstance(v, (list, tuple)):
                        return [str(i).strip() for i in v if str(i).strip()]
                except Exception:
                    continue
        return [s] if s else []
    return []


# ---------------------------------------------------------------------- scoring
def score(path, tag):
    df = pd.read_csv(path)
    col = f"predicted_answer_{tag}"

    df["pn"] = df[col].apply(normalize_arabic_text)
    df["gn"] = df["correct_answers"].apply(parse_answers).apply(
        lambda L: [normalize_arabic_text(g) for g in L])

    df["em"] = df.apply(lambda r: float(any(r.pn == g for g in r.gn)), axis=1)
    df["em25"] = df.apply(
        lambda r: float(any(r.pn[k:] == g for g in r.gn for k in range(26))), axis=1)

    def token_f1(r):
        P, best = r.pn.split(), 0.0
        for g in r.gn:
            G = g.split()
            if not P or not G:
                continue
            common = sum(min(P.count(w), G.count(w)) for w in set(P))
            if common:
                p, rc = common / len(P), common / len(G)
                best = max(best, 2 * p * rc / (p + rc))
        return best

    df["f1"] = df.apply(token_f1, axis=1)
    df["strict"] = (df["pn"] == normalize_arabic_text(NO_ANSWER)).astype(float)
    df["flex"] = df[col].apply(is_flexible_no_answer).astype(float)

    def rank_at(r, disc):
        ids = parse_ids(r["retrieved_doc_ids"])
        for i, d in enumerate(ids[:5]):
            if d == r["document_id"]:
                return disc(i)
        return 0.0

    df["mrr5"] = df.apply(lambda r: rank_at(r, lambda i: 1.0 / (i + 1)), axis=1)
    df["ndcg5"] = df.apply(lambda r: rank_at(r, lambda i: 1.0 / np.log2(i + 2)), axis=1)
    df["model"] = tag
    return df


def parse_ids(x):
    try:
        v = ast.literal_eval(x)
        return list(v) if isinstance(v, (list, tuple)) else []
    except Exception:
        return []


# ------------------------------------------------------------------ inference
def permutation_test(x, y, B=B, seed=SEED):
    rng = np.random.default_rng(seed)
    x, y = np.asarray(x, float), np.asarray(y, float)
    obs = x.mean() - y.mean()
    pool = np.concatenate([x, y])
    n, hits = len(x), 0
    for _ in range(B):
        rng.shuffle(pool)
        if abs(pool[:n].mean() - pool[n:].mean()) >= abs(obs) - 1e-12:
            hits += 1
    null_support = len(np.unique(np.round(
        [pool[:n].mean() - pool[n:].mean() for _ in range(0)] or [obs], 6)))
    return obs, (hits + 1) / (B + 1), null_support


def bootstrap_ci(v, B=B, seed=SEED):
    rng = np.random.default_rng(seed)
    v = np.asarray(v, float)
    if v.size == 0:
        return (np.nan, np.nan)
    d = rng.choice(v, size=(B, v.size), replace=True).mean(axis=1)
    return tuple(np.percentile(d, [2.5, 97.5]))


# ----------------------------------------------------------------------- main
def main():
    frames = []
    for tag in MODELS:
        p = PRED_DIR / f"predictions_{tag}_1000.csv"
        if p.exists():
            frames.append(score(p, tag))
        else:
            print(f"[skip] missing {p}")
    if not frames:
        raise SystemExit("No prediction files found.")
    allf = pd.concat(frames, ignore_index=True)

    print("=" * 78)
    print("VALIDATION — recomputed vs published")
    print("=" * 78)
    for tag in allf.model.unique():
        m = allf[allf.model == tag]
        a, u = m[~m.is_impossible], m[m.is_impossible]
        got = (100 * a.em.mean(), 100 * a.em25.mean(), a.f1.mean(),
               100 * u.strict.mean(), 100 * u["flex"].mean())
        exp = PUBLISHED[tag]
        ok = all(abs(g - e) < 0.02 for g, e in zip(got, exp))
        print(f"{MODELS[tag]:26s} EM {got[0]:5.2f}/{exp[0]:<5.2f} "
              f"EM25 {got[1]:5.2f}/{exp[1]:<5.2f} F1 {got[2]:.4f}/{exp[2]:.4f} "
              f"strict {got[3]:5.2f}/{exp[3]:<5.2f} flex {got[4]:5.2f}/{exp[4]:<5.2f} "
              f"{'OK' if ok else '*** MISMATCH ***'}")

    print("\n" + "=" * 78)
    print("BREAKDOWN BY SPLIT ORIGIN")
    print("=" * 78)
    rows = []
    for tag in allf.model.unique():
        m = allf[allf.model == tag]
        for s in ["train", "validation", "test"]:
            a, u = m[(m.split == s) & ~m.is_impossible], m[(m.split == s) & m.is_impossible]
            rows.append({"model": MODELS[tag], "split": s,
                         "n_ans": len(a), "n_unans": len(u),
                         "EM": 100 * a.em.mean(), "EM25": 100 * a.em25.mean(),
                         "token_F1": a.f1.mean(),
                         "abst_strict": 100 * u.strict.mean(),
                         "abst_flex": 100 * u["flex"].mean(),
                         "MRR5": m[m.split == s].mrr5.mean(),
                         "nDCG5": m[m.split == s].ndcg5.mean()})
    bd = pd.DataFrame(rows)
    print(bd.round(4).to_string(index=False))
    bd.to_csv(OUT_DIR / "breakdown_by_split.csv", index=False)

    print("\n" + "=" * 78)
    print(f"TRAIN-ORIGIN vs TEST-ORIGIN  (permutation, B={B:,})")
    print("=" * 78)
    specs = [("token-F1 (ans)", "f1", False), ("EM (ans)", "em", False),
             ("EM25 (ans)", "em25", False), ("strict abst (unans)", "strict", True),
             ("flex abst (unans)", "flex", True), ("MRR@5 (all)", "mrr5", None)]
    out = []
    for tag in allf.model.unique():
        m = allf[allf.model == tag]
        for label, col, imp in specs:
            sub = m if imp is None else m[m.is_impossible == imp]
            x = sub[sub.split == "train"][col].values
            y = sub[sub.split == "test"][col].values
            d, p, _ = permutation_test(x, y)
            lo, hi = bootstrap_ci(y)
            out.append({"model": MODELS[tag], "metric": label,
                        "train": x.mean(), "n_train": len(x),
                        "test": y.mean(), "n_test": len(y),
                        "delta": d, "p_perm": p,
                        "test_ci_lo": lo, "test_ci_hi": hi})
            print(f"{MODELS[tag]:26s} {label:22s} train={x.mean():.4f}(n={len(x):3d}) "
                  f"test={y.mean():.4f}(n={len(y):3d})  d={d:+.4f}  p={p:.3f}")
    pd.DataFrame(out).to_csv(OUT_DIR / "train_vs_test_tests.csv", index=False)
    print(f"\nWrote {OUT_DIR}/breakdown_by_split.csv and train_vs_test_tests.csv")
    print("Reminder: do not report the EM/EM25 p-values as evidence of no "
          "difference — see the POWER WARNING at the top of this file.")


if __name__ == "__main__":
    main()

VALIDATION — recomputed vs published
LLaMA 3 8B Instruct        EM  3.20/3.20  EM25  5.00/5.00  F1 0.1757/0.1757 strict  7.60/7.60  flex 12.80/12.80 OK
Command-R7B-12-2024        EM  1.20/1.20  EM25  2.20/2.20  F1 0.1625/0.1625 strict 53.80/53.80 flex 54.20/54.20 OK
Mistral-7B-Instruct-v0.1   EM  0.60/0.60  EM25  0.80/0.80  F1 0.0863/0.0863 strict  7.60/7.60  flex 56.60/56.60 OK

BREAKDOWN BY SPLIT ORIGIN
                   model      split  n_ans  n_unans     EM   EM25  token_F1  abst_strict  abst_flex   MRR5  nDCG5
     LLaMA 3 8B Instruct      train    339      362 3.2448 5.0147    0.1720       7.1823    12.9834 0.4788 0.5036
     LLaMA 3 8B Instruct validation     83       65 2.4096 4.8193    0.1816       9.2308    10.7692 0.4458 0.4742
     LLaMA 3 8B Instruct       test     78       73 3.8462 5.1282    0.1852       8.2192    13.6986 0.4488 0.4760
     Command-R7B-12-2024      train    339      362 1.1799 1.7699    0.1546      54.4199    54.9724 0.4788 0.5036
     Command-R7B-12-2

In [2]:
"""
Split-overlap diagnostic for the ArabicaQA MRC evaluation subset.

Answers two questions needed for Reviewer #1, Concern #1:
  (A) Does ArabicaQA partition MRC at the document level or the question level?
      i.e. does any document_id appear in more than one of train/dev/test?
  (B) What is the split composition of the 1,000-question evaluation subset,
      and is the split label usable for a post-hoc sensitivity analysis?

Run from the same working directory as the sampling notebook.
"""

import json
from pathlib import Path

import pandas as pd

FULL_PATH = Path("arabicaqa_rag_results/dataset/df_all_mrc.csv")
SAMPLE_PATH = Path("arabicaqa_rag_results/dataset/df_sample_1000.csv")

pd.set_option("display.width", 120)


def rule(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


# ----------------------------------------------------------------------
# 0) Load
# ----------------------------------------------------------------------
df_all = pd.read_csv(FULL_PATH)
df_sample = pd.read_csv(SAMPLE_PATH)

for df in (df_all, df_sample):
    df["is_impossible"] = df["is_impossible"].astype(bool)
    df["split"] = df["split"].astype(str).str.strip().str.lower()

rule("0) Shapes and columns")
print("Full MRC frame :", df_all.shape, "|", list(df_all.columns))
print("Sample frame   :", df_sample.shape, "|", list(df_sample.columns))


# ----------------------------------------------------------------------
# 1) Split sizes — cross-check against Table 1 of the manuscript
# ----------------------------------------------------------------------
rule("1) Split sizes vs. manuscript Table 1")
tab = (
    df_all.groupby(["split", "is_impossible"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={False: "answerable", True: "unanswerable"})
)
tab["total"] = tab.sum(axis=1)
print(tab)
print("\nOverall unanswerable share: "
      f"{100 * df_all['is_impossible'].mean():.2f}%   (manuscript reports 3.99%)")
print("Expected from Table 1 -> train 62186/2596, dev 13483/561, test 13426/544")


# ----------------------------------------------------------------------
# 2) THE MAIN CHECK: does any document_id span more than one split?
# ----------------------------------------------------------------------
rule("2) document_id overlap across splits (FULL frame)")
doc_splits = df_all.groupby("document_id")["split"].nunique()
n_docs = len(doc_splits)
n_multi = int((doc_splits > 1).sum())

print(f"Unique document_id values : {n_docs:,}")
print(f"Documents in >1 split     : {n_multi:,}  ({100 * n_multi / n_docs:.2f}%)")

if n_multi == 0:
    print("\n=> DOCUMENT-LEVEL PARTITION. No document is shared between splits.")
else:
    print("\n=> QUESTION-LEVEL PARTITION. Documents are reused across splits.")
    combos = (
        df_all.groupby("document_id")["split"]
        .apply(lambda s: "+".join(sorted(set(s))))
    )
    print("\nSplit-combination frequency over documents:")
    print(combos.value_counts())

    shared = doc_splits[doc_splits > 1].index
    n_q_affected = int(df_all["document_id"].isin(shared).sum())
    print(f"\nQuestions attached to a shared document: "
          f"{n_q_affected:,} / {len(df_all):,} "
          f"({100 * n_q_affected / len(df_all):.2f}%)")

    print("\nExample shared documents (first 5):")
    for did in list(shared)[:5]:
        sub = df_all[df_all["document_id"] == did]
        print(f"  doc {did}: splits={sorted(set(sub['split']))}, n_questions={len(sub)}")


# ----------------------------------------------------------------------
# 3) Is document_id -> context a 1:1 mapping?
#    Matters because the index is built by deduplicating on document_id.
# ----------------------------------------------------------------------
rule("3) document_id -> context consistency")
ctx_per_doc = df_all.groupby("document_id")["context"].nunique()
n_inconsistent = int((ctx_per_doc > 1).sum())
print(f"Documents with >1 distinct context string: {n_inconsistent:,}")
if n_inconsistent:
    print("WARNING: deduplicating on document_id will silently drop context variants.")
    print(ctx_per_doc[ctx_per_doc > 1].head(10))
else:
    print("=> Clean. Deduplication on document_id is lossless.")


# ----------------------------------------------------------------------
# 4) Split composition of the 1,000-question evaluation subset
# ----------------------------------------------------------------------
rule("4) Split composition of the evaluation subset")
comp = (
    df_sample.groupby(["split", "is_impossible"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={False: "answerable", True: "unanswerable"})
)
comp["total"] = comp.sum(axis=1)
comp["pct_of_1000"] = (100 * comp["total"] / len(df_sample)).round(1)
print(comp)

print("\nExpected under proportional sampling from the merged pool:")
for split in sorted(df_all["split"].unique()):
    for imp, label, n_draw in [(False, "answerable", 500), (True, "unanswerable", 500)]:
        pool = df_all[df_all["is_impossible"] == imp]
        share = (pool["split"] == split).mean()
        print(f"  {split:5s} {label:12s}: ~{share * n_draw:5.1f}")


# ----------------------------------------------------------------------
# 5) Does the subset itself contain cross-split documents?
# ----------------------------------------------------------------------
rule("5) Document reuse inside the evaluation subset")
print(f"Unique document_id in subset : {df_sample['document_id'].nunique():,} / {len(df_sample):,}")
dup = df_sample["document_id"].value_counts()
print(f"Documents backing >1 sampled question: {int((dup > 1).sum()):,}")

sample_docs = set(df_sample["document_id"])
test_sample_docs = set(df_sample.loc[df_sample["split"] == "test", "document_id"])
train_docs_all = set(df_all.loc[df_all["split"] == "train", "document_id"])
leak = test_sample_docs & train_docs_all
print(f"\nSampled test-origin documents that also appear in train (full frame): "
      f"{len(leak):,} / {len(test_sample_docs):,}")
print("(If >0, a test-only re-run would still retrieve from train-linked passages.)")


# ----------------------------------------------------------------------
# 6) Persist a compact join key for the sensitivity analysis
# ----------------------------------------------------------------------
rule("6) Writing split label lookup")
out = df_sample[["question_id", "split", "is_impossible", "document_id"]].copy()
out_path = Path("arabicaqa_rag_results/dataset/sample_split_labels.csv")
out.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Wrote {out_path} ({len(out)} rows).")
print("Join this to the per-model result files on question_id to score by split origin.")

summary = {
    "n_documents_total": int(n_docs),
    "n_documents_multi_split": int(n_multi),
    "partition_level": "document" if n_multi == 0 else "question",
    "unanswerable_share_full": round(float(df_all["is_impossible"].mean()), 4),
    "subset_split_counts": comp["total"].to_dict(),
}
Path("arabicaqa_rag_results/dataset/split_overlap_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Wrote split_overlap_summary.json")


0) Shapes and columns
Full MRC frame : (92796, 7) | ['split', 'document_id', 'question_id', 'question', 'context', 'answers', 'is_impossible']
Sample frame   : (1000, 7) | ['split', 'document_id', 'question_id', 'question', 'context', 'answers', 'is_impossible']

1) Split sizes vs. manuscript Table 1
is_impossible  answerable  unanswerable  total
split                                         
test                13426           544  13970
train               62186          2596  64782
validation          13483           561  14044

Overall unanswerable share: 3.99%   (manuscript reports 3.99%)
Expected from Table 1 -> train 62186/2596, dev 13483/561, test 13426/544

2) document_id overlap across splits (FULL frame)
Unique document_id values : 14,051
Documents in >1 split     : 0  (0.00%)

=> DOCUMENT-LEVEL PARTITION. No document is shared between splits.

3) document_id -> context consistency
Documents with >1 distinct context string: 0
=> Clean. Deduplication on document_id is lossle